In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV, KFold, cross_val_predict

from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score, roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.under_sampling import RandomUnderSampler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

import warnings
warnings.filterwarnings("ignore")

# -------------------------------
# 1. 데이터 전처리
df = pd.read_csv("df_concat_BA1.csv")  ## 통합 전처리한 파일명 삽입.
target_col = '기부여부'
df = df.drop(columns=df.columns[df.isnull().mean() >= 0.5])
df = pd.DataFrame(SimpleImputer(strategy='most_frequent').fit_transform(df), columns=df.columns)

df[target_col] = df[target_col].astype(int)
X = df.drop(columns=[target_col])
y = df[target_col]

low_cardinality_cols = X.columns[X.nunique() <= 10]
X = pd.get_dummies(X, columns=low_cardinality_cols, drop_first=True)

X = pd.DataFrame(VarianceThreshold(threshold=0.01).fit_transform(X))
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
drop_cols = [col for col in upper.columns if any(upper[col] > 0.9)]
X.drop(X.columns[drop_cols], axis=1, inplace=True)

# -------------------------------
# 2. 평가 함수
def evaluate_model(name, y_true, y_pred, y_proba, fold):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp)
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_proba)
    return {
        "Fold": fold,
        "Algorithm": name,
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "AUC": auc
    }

# -------------------------------
# 3. 파라미터 튜닝용 
param_grids = {
    "Logistic Regression": {'C': [0.1, 1, 10]},
    "Random Forest": {'n_estimators': [100, 200], 'max_depth': [5, 10]},
    "XGBoost": {'n_estimators': [100], 'max_depth': [3, 5]},
    "LightGBM": {'n_estimators': [100], 'num_leaves': [31, 50]},
    "CatBoost": {'depth': [4, 6], 'iterations': [100], 'learning_rate': [0.1]},
    "MLP": {'hidden_layer_sizes': [(64,), (64, 32)], 'alpha': [0.0001, 0.001]}
}

base_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "LightGBM": LGBMClassifier(random_state=42),
    "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
    "MLP": MLPClassifier(max_iter=300, random_state=42)
}

# -------------------------------
# 4. 학습 + 평가
results = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_valid_scaled = scaler.transform(X_valid)

    rus = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
    X_res, y_res = rus.fit_resample(X_train_scaled, y_train)

    for name, model in base_models.items():
        clf = GridSearchCV(model, param_grids[name], cv=3, scoring='f1', n_jobs=-1)
        clf.fit(X_res, y_res)
        best_model = clf.best_estimator_
        y_pred = best_model.predict(X_valid_scaled)
        y_proba = best_model.predict_proba(X_valid_scaled)[:, 1]
        results.append(evaluate_model(name, y_valid, y_pred, y_proba, fold))

    # CNN
    cnn = Sequential([
        Dense(64, activation='relu', input_shape=(X_res.shape[1],)),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    cnn.fit(X_res, y_res, epochs=10, batch_size=32, verbose=0)
    y_pred_cnn = cnn.predict(X_valid_scaled).flatten()
    y_pred_cnn_label = (y_pred_cnn >= 0.5).astype(int)
    results.append(evaluate_model("CNN", y_valid, y_pred_cnn_label, y_pred_cnn, fold))

# -------------------------------
# 5. KNN 평가 (언더샘플링 포함)
X_scaled_all = MinMaxScaler().fit_transform(X)
X_res_all, y_res_all = RandomUnderSampler(sampling_strategy=0.5, random_state=42).fit_resample(X_scaled_all, y)

knn = KNeighborsClassifier(n_neighbors=5)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
y_pred_knn = cross_val_predict(knn, X_res_all, y_res_all, cv=kf, method='predict')
y_proba_knn = cross_val_predict(knn, X_res_all, y_res_all, cv=kf, method='predict_proba')[:, 1]
results.append(evaluate_model("KNN", y_res_all, y_pred_knn, y_proba_knn, fold='All'))

# -------------------------------
# 6. 결과 정리
result_df = pd.DataFrame(results)

# 평균 성능 테이블
summary = result_df.groupby("Algorithm")[[
    "Precision", "Recall", "Specificity", "F1-score", "Accuracy", "Balanced Accuracy", "AUC"
]].mean().round(4).sort_values("AUC", ascending=False)

# 출력
print(" 전체 Fold 결과:")
display(result_df)

print("\n 모델별 평균 성능 요약:")
display(summary)

In [ ]:
# -------------------------------
# 7. 최종 테스트 성능 평가용 데이터 분할
from sklearn.model_selection import train_test_split

# 전체를 다시 Train-Test split
X_train_full, X_test_final, y_train_full, y_test_final = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling
scaler_final = MinMaxScaler()
X_train_scaled_final = scaler_final.fit_transform(X_train_full)
X_test_scaled_final = scaler_final.transform(X_test_final)

# Undersampling
rus_final = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
X_train_res_final, y_train_res_final = rus_final.fit_resample(X_train_scaled_final, y_train_full)

# 최종 결과 저장 리스트
final_test_results = []

# 튜닝된 모델로 다시 학습 후 테스트셋 평가
for name, model in base_models.items():
    clf_final = GridSearchCV(model, param_grids[name], cv=3, scoring='f1', n_jobs=-1)
    clf_final.fit(X_train_res_final, y_train_res_final)
    best_model = clf_final.best_estimator_
    y_pred_final = best_model.predict(X_test_scaled_final)
    y_proba_final = best_model.predict_proba(X_test_scaled_final)[:, 1]
    final_test_results.append(evaluate_model(name, y_test_final, y_pred_final, y_proba_final, fold="FinalTest"))

# CNN도 동일하게
cnn_final = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_res_final.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])
cnn_final.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn_final.fit(X_train_res_final, y_train_res_final, epochs=10, batch_size=32, verbose=0)
y_pred_cnn_final = cnn_final.predict(X_test_scaled_final).flatten()
y_pred_cnn_final_label = (y_pred_cnn_final >= 0.5).astype(int)
final_test_results.append(evaluate_model("CNN", y_test_final, y_pred_cnn_final_label, y_pred_cnn_final, fold="FinalTest"))

# KNN도 동일하게 (학습 데이터로 학습, 테스트셋 평가)
knn_final = KNeighborsClassifier(n_neighbors=5)
knn_final.fit(X_train_res_final, y_train_res_final)
y_pred_knn_final = knn_final.predict(X_test_scaled_final)
y_proba_knn_final = knn_final.predict_proba(X_test_scaled_final)[:, 1]
final_test_results.append(evaluate_model("KNN", y_test_final, y_pred_knn_final, y_proba_knn_final, fold="FinalTest"))

# -------------------------------
# 8. 최종 테스트 결과 출력
final_test_df = pd.DataFrame(final_test_results)
final_test_summary = final_test_df[[
    "Algorithm", "Precision", "Recall", "Specificity", "F1-score", "Accuracy", "Balanced Accuracy", "AUC"
]].round(4).sort_values("AUC", ascending=False)

print("최종 테스트 결과:")
display(final_test_df)

print("\n최종 테스트 성능 요약:")
display(final_test_summary)
